# 10 Visualisera och nyansera

Nu har vi en validerad tabell där fyra öppna källor möts. I den här notebooken använder vi den för att besvara en fråga, och lika viktigt, för att se vad datan inte kan svara på.

En analys börjar med ett behov, inte med datan. Vi tar ett exempel.

> Hur upplever invånarna hastigheten på vår kommuns webbplats, och hur står vi oss mot kommuner som liknar vår?

Ingen enskild källa besvarar det. Webperf testar webbplatsen men mäter inte riktiga besökare. CrUX mäter riktiga besökare men vet inget om kommunen. Wikidata vet hur stor kommunen är och vilket län den ligger i. Kolada vet hur snabb uppkoppling invånarna har. Svaret kräver att vi korsrefererar källorna, alltså låter varje källa bidra med det den är bra på och prövar dem mot varandra.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("..") / "data"
FIGURER = DATA / "figurer"
FIGURER.mkdir(parents=True, exist_ok=True)

webbplatser = pd.read_parquet(DATA / "harmoniserad" / "webbplatser.parquet")

## Vilken kommun gäller det?

Vi väljer en kommun att utgå från. Byt namnet och kör om notebooken för att göra samma analys för en annan kommun.

In [ ]:
EXEMPEL = "Ale kommun"

webbplatser.loc[webbplatser["namn"] == EXEMPEL].T.head(12)

`.T` vänder tabellen så att kolumnerna blir rader, vilket gör en enda rad lättare att läsa.

Vi jämför bara kommuner med kommuner. Regionernas webbplatser har andra uppdrag och andra besökare, så att blanda in dem skulle göra jämförelsen mindre rättvis.

In [ ]:
kommuner = webbplatser[webbplatser["typ"] == "kommun"].copy()
exempel = kommuner[kommuner["namn"] == EXEMPEL]

len(kommuner), len(exempel)

## Ett gemensamt utseende för diagrammen

Vi ställer in utseendet en gång, så att alla diagram ser likadana ut. Alla kommuner ritas i blått och den kommun vi utgår från i orange. Rutnätet är ljust, ligger bakom datan och tar inte uppmärksamhet från den. `rcParams` är matplotlibs inställningar. `ETIKETT` ger text i diagrammen en bakgrund i samma färg som ytan, så att den går att läsa även ovanpå punkter och staplar.

In [ ]:
BLA = "#2a78d6"
ORANGE = "#eb6834"
YTA = "#fcfcfb"
TEXT = "#0b0b0b"
TEXT_SVAG = "#52514e"
RUTNAT = "#e4e3df"
ETIKETT = {"facecolor": YTA, "edgecolor": "none", "pad": 2}

plt.rcParams.update({
    "figure.facecolor": YTA,
    "axes.facecolor": YTA,
    "axes.edgecolor": RUTNAT,
    "axes.labelcolor": TEXT_SVAG,
    "axes.titlecolor": TEXT,
    "axes.titlesize": 13,
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": RUTNAT,
    "grid.linewidth": 0.8,
    "xtick.color": TEXT_SVAG,
    "ytick.color": TEXT_SVAG,
    "font.size": 10,
})

## 1. Säger två mätmetoder samma sak?

Webperf ger ett prestandabetyg mellan 1 och 5 efter att ha testat startsidan med automatiska verktyg. CrUX ger LCP, hur lång tid det tar innan det största innehållet syns, för riktiga besökare på hela webbplatsen. Om båda mäter hur snabb webbplatsen är borde kommuner med högt betyg ha låg LCP.

Vi ritar ett spridningsdiagram med en punkt per kommun. `scatter` ritar punkterna och `annotate` skriver namnet vid den kommun vi utgår från.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(kommuner["betyg_prestanda"], kommuner["lcp_p75_ms"], s=28, color=BLA, edgecolor=YTA, linewidth=1, label="Kommuner")
ax.scatter(exempel["betyg_prestanda"], exempel["lcp_p75_ms"], s=60, color=ORANGE, edgecolor=YTA, linewidth=1.5, label=EXEMPEL)
ax.annotate(EXEMPEL, (exempel["betyg_prestanda"].iloc[0], exempel["lcp_p75_ms"].iloc[0]),
            xytext=(10, -16), textcoords="offset points", color=TEXT, bbox=ETIKETT)

ax.set_title("Webperfs prestandabetyg och LCP från riktiga besökare")
ax.set_xlabel("Prestandabetyg i Webperf (1 till 5, högre är bättre)")
ax.set_ylabel("LCP p75 i CrUX, millisekunder (lägre är bättre)")
ax.legend(frameon=False)

fig.savefig(FIGURER / "betyg-och-lcp.png", dpi=200, bbox_inches="tight")
plt.show()

Punkterna bildar ett moln utan tydlig riktning. Det kan vi mäta med en korrelation, ett tal mellan −1 och 1. Nära 1 betyder att höga värden på det ena hör ihop med höga på det andra, nära −1 det omvända och nära 0 att det inte finns något tydligt samband.

Vi använder Spearmans korrelation, som jämför i vilken ordning kommunerna hamnar i stället för de exakta värdena. Den påverkas mindre av enstaka extrema kommuner. Vi räknar sambandet mellan LCP och några andra kolumner från olika källor.

In [ ]:
jamfor = ["betyg_prestanda", "betyg_totalt", "bredband_ned_mbit_s", "befolkning", "inp_p75_ms"]

samband = kommuner[jamfor + ["lcp_p75_ms"]].astype(float).corr(method="spearman")["lcp_p75_ms"]

samband.drop("lcp_p75_ms").round(2).sort_values()

`astype(float)` gör om alla kolumner till decimaltal, eftersom `corr` inte hanterar alla typer. Kommuner som saknar ett värde räknas inte med.

När vi körde notebooken i oktober 2026 var sambandet mellan prestandabetyget och LCP nästan noll. Det betyder inte att någon av källorna har fel. De mäter olika saker.

- Webperf testar en sida, startsidan, vid ett tillfälle och under samma förhållanden för alla kommuner.
- CrUX samlar alla sidvisningar under 28 dagar, på alla sidor, med besökarnas egna enheter och uppkopplingar.

Totalbetyget, som också väger in tillgänglighet, säkerhet och standarder, hade ett tydligare samband med LCP. Det säger något om att webbplatser som sköts väl på flera områden ofta också är snabba för besökarna, men inte varför.

## 2. Klarar kommunerna Googles gränser?

Google anger gränser för vad som är bra. LCP ska vara högst 2,5 sekunder, INP högst 200 millisekunder och CLS högst 0,1. Vi räknar hur stor andel av kommunerna som klarar varje gräns, bland dem som har ett värde.

In [ ]:
granser = {"lcp_p75_ms": 2500, "inp_p75_ms": 200, "cls_p75": 0.1}

andel_bra = {}
for kolumn, grans in granser.items():
    varden = kommuner[kolumn].dropna()
    andel_bra[kolumn] = round((varden <= grans).mean() * 100, 1)

pd.Series(andel_bra, name="procent som klarar gränsen")

`mean` på en kolumn med `True` och `False` ger andelen `True`, eftersom `True` räknas som 1.

Nästan alla kommuner klarar gränserna. Det är goda nyheter, men gör också gränserna nästan värdelösa för att jämföra kommuner med varandra. En gräns delar in i två grupper och döljer skillnaderna inom gruppen. Därför tittar vi på hela fördelningen i stället.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(kommuner["lcp_p75_ms"].dropna(), bins=30, color=BLA, edgecolor=YTA, linewidth=1)
ax.set_ylim(top=ax.get_ylim()[1] * 1.15)
ax.axvline(2500, color=TEXT_SVAG, linewidth=1)
ax.annotate("Googles gräns för bra, 2 500 ms", (2500, ax.get_ylim()[1] * 0.97), xytext=(-6, 0),
            textcoords="offset points", ha="right", va="top", color=TEXT_SVAG, bbox=ETIKETT)
ax.axvline(exempel["lcp_p75_ms"].iloc[0], color=ORANGE, linewidth=2)
ax.annotate(EXEMPEL, (exempel["lcp_p75_ms"].iloc[0], ax.get_ylim()[1] * 0.97), xytext=(6, 0),
            textcoords="offset points", va="top", color=TEXT, bbox=ETIKETT)

ax.set_title("LCP för kommunernas webbplatser")
ax.set_xlabel("LCP p75 i CrUX, millisekunder")
ax.set_ylabel("Antal kommuner")
ax.grid(axis="x", visible=False)

fig.savefig(FIGURER / "lcp-fordelning.png", dpi=200, bbox_inches="tight")
plt.show()

Ett histogram delar in värdena i intervall, `bins`, och visar hur många kommuner som hamnar i varje. Med `set_ylim` ger vi diagrammet lite luft ovanför de högsta staplarna, så att etiketterna får plats. De flesta kommuner ligger långt under gränsen, men spridningen är stor. Var vår kommun hamnar i fördelningen säger mer än att den klarar gränsen.

## 3. Spelar invånarnas uppkoppling roll?

Fältdata påverkas av besökarnas uppkoppling, labbdata gör det inte. Vi korsrefererar CrUX med Bredbandskollens mätningar från Kolada.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(kommuner["bredband_ned_mbit_s"], kommuner["lcp_p75_ms"], s=28, color=BLA, edgecolor=YTA, linewidth=1, label="Kommuner")
ax.scatter(exempel["bredband_ned_mbit_s"], exempel["lcp_p75_ms"], s=60, color=ORANGE, edgecolor=YTA, linewidth=1.5, label=EXEMPEL)
ax.annotate(EXEMPEL, (exempel["bredband_ned_mbit_s"].iloc[0], exempel["lcp_p75_ms"].iloc[0]),
            xytext=(10, -16), textcoords="offset points", color=TEXT, bbox=ETIKETT)

ax.set_title("Invånarnas uppkoppling och LCP")
ax.set_xlabel("Bredbandskollen, genomsnitt nedströms 2025, Mbit/s")
ax.set_ylabel("LCP p75 i CrUX, millisekunder")
ax.legend(frameon=False)

fig.savefig(FIGURER / "bredband-och-lcp.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
kommuner[["bredband_ned_mbit_s", "lcp_p75_ms", "betyg_prestanda"]].astype(float).corr(method="spearman").round(2)

Sambandet mellan uppkoppling och LCP är svagt men går åt det håll vi väntar oss, snabbare uppkoppling och lägre LCP. Mellan uppkoppling och Webperfs betyg finns inget samband, vilket stämmer med att labbtestet inte använder invånarnas uppkoppling.

Här finns flera förbehåll.

- Bredbandskollen mäter bara dem som själva väljer att testa sin uppkoppling, och testet görs i en webbläsare, oftast på en dator. Många besök på kommunens webbplats sker från mobilen.
- Besökarna på en kommuns webbplats bor inte alla i kommunen.
- Ett samband mellan medelvärden för kommuner säger inget säkert om enskilda besökare. Att dra sådana slutsatser kallas ekologiskt felslut.

## 4. Hur står vi oss mot liknande kommuner?

En liten kommun och en storstad har olika förutsättningar. Wikidata ger oss befolkning och län, så att vi kan jämföra med kommuner som liknar vår. Vi delar in kommunerna i storleksgrupper med `pd.cut`, som lägger varje värde i ett intervall.

In [ ]:
kommuner["storlek"] = pd.cut(
    kommuner["befolkning"],
    bins=[0, 10_000, 25_000, 50_000, 100_000, 10_000_000],
    labels=["under 10 000", "10 000 till 25 000", "25 000 till 50 000", "50 000 till 100 000", "över 100 000"],
)

kommuner.groupby("storlek", observed=True).agg(
    kommuner=("namn", "size"),
    lcp_median_ms=("lcp_p75_ms", "median"),
    betyg_prestanda_median=("betyg_prestanda", "median"),
    bredband_median=("bredband_ned_mbit_s", "median"),
)

Understrecken i `10_000` gör bara talet lättare att läsa. `agg` räknar flera sammanfattningar på en gång, här antal och median.

Nu sätter vi ihop en profil för vår kommun. För varje mått visar vi kommunens värde, medianen bland kommunerna i samma län och medianen i samma storleksgrupp, och vilken källa värdet kommer från.

In [ ]:
rad = kommuner[kommuner["namn"] == EXEMPEL].iloc[0]
samma_lan = kommuner[kommuner["lan"] == rad["lan"]]
samma_storlek = kommuner[kommuner["storlek"] == rad["storlek"]]

matt = [
    ("betyg_totalt", "Webperf"),
    ("betyg_prestanda", "Webperf"),
    ("lcp_p75_ms", "CrUX"),
    ("inp_p75_ms", "CrUX"),
    ("cls_p75", "CrUX"),
    ("bredband_ned_mbit_s", "Kolada"),
]

profil = pd.DataFrame([
    {
        "matt": kolumn,
        "kalla": kalla,
        EXEMPEL: rad[kolumn],
        f"median {rad['lan']}": samma_lan[kolumn].median(),
        f"median {rad['storlek']} invånare": samma_storlek[kolumn].median(),
    }
    for kolumn, kalla in matt
])

print(EXEMPEL, "har", rad["befolkning"], "invånare och ligger i", rad["lan"])
profil.round(2)

`iloc[0]` tar den första och enda raden som en serie, så att vi kan hämta värden med kolumnnamnet.

Till sist ritar vi alla kommuner i samma län, sorterade efter LCP. `barh` ritar liggande staplar, som gör namnen lätta att läsa.

In [ ]:
lanet = samma_lan.dropna(subset=["lcp_p75_ms"]).sort_values("lcp_p75_ms")
farger = [ORANGE if namn == EXEMPEL else BLA for namn in lanet["namn"]]

fig, ax = plt.subplots(figsize=(8, 0.3 * len(lanet) + 1))

ax.barh(lanet["namn"], lanet["lcp_p75_ms"], color=farger, height=0.7)
ax.invert_yaxis()
ax.margins(y=0.01)

ax.set_title(f"LCP för kommunerna i {rad['lan']}")
ax.set_xlabel("LCP p75 i CrUX, millisekunder (lägre är bättre)")
ax.grid(axis="y", visible=False)

fig.savefig(FIGURER / "lcp-i-lanet.png", dpi=200, bbox_inches="tight")
plt.show()

`invert_yaxis` vänder ordningen så att den snabbaste kommunen hamnar överst, och `margins` minskar det tomma utrymmet ovanför och under staplarna.

## Vad datan inte säger

Ett kombinerat underlag är starkare än en enskild källa, men har också fler förbehåll. Dem behöver vi redovisa tillsammans med resultatet.

- **CrUX mäter bara Chrome.** Besökare i Safari och Firefox finns inte med, och det gäller en stor del av alla mobiler.
- **Saknad data betyder inte långsam webbplats.** Solnas webbplats saknas i CrUX för att startsidan inte får indexeras, inte för att den är långsam. Några små kommuner saknar enskilda mått för att för få besökare har mätts.
- **Källorna mäter vid olika tider.** Webperfs betyg och CrUX gäller hösten 2026, befolkningen mitten av 2026 och bredbandet hela 2025.
- **p75 är inte den typiska besökaren.** Värdet säger att tre av fyra sidvisningar var minst så bra. Den sämsta fjärdedelen syns inte.
- **Labbtestet gäller startsidan.** Fältdatan gäller hela webbplatsen. Det är en av förklaringarna till att de två inte hänger ihop.
- **Samband är inte orsak.** Att två mått följs åt betyder inte att det ena orsakar det andra.

## Det här vet vi nu

- En analys börjar med ett behov, och behovet avgör vilka källor och kolumner vi behöver.
- Att korsreferera betyder att varje källa bidrar med det den är bra på och att vi prövar källorna mot varandra.
- Två mätmetoder av samma sak kan ge olika svar, och skillnaden säger något om vad de mäter.
- En gräns som nästan alla klarar säger lite. Hela fördelningen säger mer.
- Jämförelser blir rättvisare mot liknande kommuner, i samma län eller av samma storlek.
- Förbehållen hör till resultatet.

Diagrammen finns sparade i mappen `data/figurer`. I nästa notebook samlar vi stegen till en pipeline som kan köras från början till slut utan att öppna en notebook.